# Regenerate TPAMI revision figures (except 01_training_comparison and 09_layer_entropy)

This notebook regenerates the manuscript/revision figures from the unified `regenerate_revision_figures.py` script, including `fig_main_mi_control`, `07_noise_ablation`, and `figS3_distance_distortion`.


## 1. Mount Drive



In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


## 2. Verify GPU



In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 3. Stage scripts from Drive



In [3]:
import os, shutil, subprocess

DRIVE_BASE = "/content/drive/MyDrive"
SEARCH_PATHS = [
    os.path.join(DRIVE_BASE, "pe_experiment"),
    DRIVE_BASE,
]

def find_on_drive(fname):
    for base in SEARCH_PATHS:
        p = os.path.join(base, fname)
        if os.path.isfile(p):
            return p
    return None

# Required scripts for final figure regeneration.
# regenerate_revision_figures.py is the single figure-generation entry point.
# The other scripts are helper dependencies used by that entry point.
required_scripts = [
    "regenerate_revision_figures.py",
    "full_scale_experiment.py",
    "apply_2d_alibi_patch.py",
    "compute_mi_cls_controls.py",
]

for script_name in required_scripts:
    src = find_on_drive(script_name)
    if src is None:
        raise FileNotFoundError(
            f"Cannot find required script: {script_name} in {SEARCH_PATHS}"
        )

    dst = f"/content/{script_name}"
    shutil.copy(src, dst)
    print(
        f"Copied: {src} -> {dst} "
        f"({os.path.getsize(dst) / 1024:.1f} KB)"
    )

print("\nApplying 2D-ALiBi patch...")
result = subprocess.run(
    "cd /content && python apply_2d_alibi_patch.py "
    "full_scale_experiment.py full_scale_experiment_v2.py",
    shell=True,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("2D-ALiBi patch failed")

print("Staging complete.")


Copied: /content/drive/MyDrive/regenerate_revision_figures.py -> /content/regenerate_revision_figures.py (62.6 KB)
Copied: /content/drive/MyDrive/pe_experiment/full_scale_experiment.py -> /content/full_scale_experiment.py (65.3 KB)
Copied: /content/drive/MyDrive/apply_2d_alibi_patch.py -> /content/apply_2d_alibi_patch.py (8.6 KB)
Copied: /content/drive/MyDrive/compute_mi_cls_controls.py -> /content/compute_mi_cls_controls.py (19.0 KB)

Applying 2D-ALiBi patch...
  [OK]   'INSERT TwoDALiBi class': inserted before
  [OK]   'MultiHeadAttention.__init__ branch': replaced
  [OK]   'MultiHeadAttention.forward attn_bias': replaced
  [OK]   'MultiHeadAttention.forward fallback': replaced
  [OK]   'VisionTransformer PE-type whitelist': replaced
  [OK]   'argparse choices': replaced

Patched file written to: full_scale_experiment_v2.py
Diff size: +2838 characters

Staging complete.


## 4. Configure Figure 3 and Figure 7 inputs

`figS3_distance_distortion` is generated analytically and does not require any external input files.


In [4]:
import os

# Figure 3 panel (a): ImageNet-100 per-layer MI aggregate JSON.
FIG3_IMAGENET_MI_JSON = (
    "/content/drive/MyDrive/revision_results/imagenet100/_aggregate.json"
)

# Figure 3 panel (b): CIFAR-100 fixed-slope 1D-vs-2D ALiBi MI summary.
FIG3_CIFAR_FIXED_MI_JSON = (
    "/content/drive/MyDrive/revision_results/mi_cls_control/"
    "cifar100_canonical_n12/paired_alibi_mi_summary.json"
)

# Figure 3 panel (b): CIFAR-100 magnitude-matched 1D-vs-2D ALiBi MI summary.
FIG3_CIFAR_MATCHED_MI_JSON = (
    "/content/drive/MyDrive/revision_results/mi_cls_control/"
    "cifar100_canonical_matched2d_n12/paired_alibi_mi_summary.json"
)

FIG3_OUTPUT_NAME = "fig_main_mi_control"
FIGURE_OUTPUT_DIR = "/content/figures_revision"

print("Figure 3 input configuration:")
print("-" * 60)
missing = []
for label, path in [
    ("ImageNet MI aggregate", FIG3_IMAGENET_MI_JSON),
    ("CIFAR fixed ALiBi MI", FIG3_CIFAR_FIXED_MI_JSON),
    ("CIFAR matched ALiBi MI", FIG3_CIFAR_MATCHED_MI_JSON),
]:
    ok = os.path.isfile(path)
    print(f"{label:26s}: {'OK' if ok else 'MISSING'}")
    print(f"  {path}")
    if ok:
        print(f"  size = {os.path.getsize(path) / 1024:.1f} KB")
    else:
        missing.append(path)

if missing:
    raise FileNotFoundError(
        "Missing required Figure 3 input file(s):\n" + "\n".join(missing)
    )


# Figure 7: saved noise-ablation / PE-removal analysis JSON.
FIG7_ANALYSIS_JSON = (
    "/content/drive/MyDrive/pe_experiment/results/analysis_data.json"
)

print(f"  Figure 7 analysis JSON     : {FIG7_ANALYSIS_JSON}")

print("Figure S3 (figS3_distance_distortion): generated analytically; no external inputs required.")


Figure 3 input configuration:
------------------------------------------------------------
ImageNet MI aggregate     : OK
  /content/drive/MyDrive/revision_results/imagenet100/_aggregate.json
  size = 5.0 KB
CIFAR fixed ALiBi MI      : OK
  /content/drive/MyDrive/revision_results/mi_cls_control/cifar100_canonical_n12/paired_alibi_mi_summary.json
  size = 57.2 KB
CIFAR matched ALiBi MI    : OK
  /content/drive/MyDrive/revision_results/mi_cls_control/cifar100_canonical_matched2d_n12/paired_alibi_mi_summary.json
  size = 61.2 KB
  Figure 7 analysis JSON     : /content/drive/MyDrive/pe_experiment/results/analysis_data.json
Figure S3 (figS3_distance_distortion): generated analytically; no external inputs required.


## 5. Generate figures

This runs the unified figure-regeneration script and produces all supported figures, including Figure 3, Figure 7, and Supplementary Figure S3.


In [5]:
import subprocess, sys, shlex

cmd_parts = [
    "python", "-u", "/content/regenerate_revision_figures.py",
    "--output-dir", FIGURE_OUTPUT_DIR,
    "--fig3-imagenet-mi-json", FIG3_IMAGENET_MI_JSON,
    "--fig3-cifar-fixed-mi-json", FIG3_CIFAR_FIXED_MI_JSON,
    "--fig3-cifar-matched-mi-json", FIG3_CIFAR_MATCHED_MI_JSON,
    "--fig3-output-name", FIG3_OUTPUT_NAME,
    "--fig7-analysis-json", FIG7_ANALYSIS_JSON,
]
cmd = "cd /content && " + " ".join(shlex.quote(x) for x in cmd_parts)

print("=" * 60)
print("Generating all revision figures (including Figure 7 and Figure S3)...")
print(cmd)
print("=" * 60)

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end="", flush=True)
    sys.stdout.flush()

exit_code = process.wait()
print()
if exit_code == 0:
    print("DONE. All figures generated.")
else:
    raise RuntimeError(f"Figure generation failed with exit code {exit_code}")


Generating all revision figures (including Figure 7 and Figure S3)...
cd /content && python -u /content/regenerate_revision_figures.py --output-dir /content/figures_revision --fig3-imagenet-mi-json /content/drive/MyDrive/revision_results/imagenet100/_aggregate.json --fig3-cifar-fixed-mi-json /content/drive/MyDrive/revision_results/mi_cls_control/cifar100_canonical_n12/paired_alibi_mi_summary.json --fig3-cifar-matched-mi-json /content/drive/MyDrive/revision_results/mi_cls_control/cifar100_canonical_matched2d_n12/paired_alibi_mi_summary.json --fig3-output-name fig_main_mi_control --fig7-analysis-json /content/drive/MyDrive/pe_experiment/results/analysis_data.json
Figures Revision - Generating figures for TPAMI submission

1. Pre-flight check: verifying checkpoints...
     [OK] learned: 344 MB
     [OK] sinusoidal: 344 MB
     [OK] rope: 344 MB
     [OK] alibi: 345 MB
     [OK] alibi_2d: 347 MB

2. Extracting PE matrices and computing metrics...
   Processing learned...
     [found PE in 

## 6. Preview PNG outputs



In [6]:
from IPython.display import Image, display
import os

FIGURE_DIR = "/content/figures_revision"
for fig_name in sorted(os.listdir(FIGURE_DIR)):
    if fig_name.endswith(".png"):
        print(f"=== {fig_name} ===")
        display(Image(os.path.join(FIGURE_DIR, fig_name), width=800))



Output hidden; open in https://colab.research.google.com to view.

## 7. Copy generated figures back to Drive



In [7]:
import os, shutil

SRC = "/content/figures_revision"
DST = "/content/drive/MyDrive/pe_experiment/figures_revision"
os.makedirs(DST, exist_ok=True)

for f in sorted(os.listdir(SRC)):
    shutil.copy(os.path.join(SRC, f), os.path.join(DST, f))
    print(f"Copied: {f}")
print(f"All figures saved to: {DST}")



Copied: 03_pca_tsne.pdf
Copied: 03_pca_tsne.png
Copied: 06_mutual_information.pdf
Copied: 06_mutual_information.png
Copied: 07_noise_ablation.pdf
Copied: 07_noise_ablation.png
Copied: fig2_cosine_similarity.pdf
Copied: fig2_cosine_similarity.png
Copied: fig4a_embedding_entropy.pdf
Copied: fig4a_embedding_entropy.png
Copied: fig4b_intrinsic_entropy.pdf
Copied: fig4b_intrinsic_entropy.png
Copied: fig5a_embedding_variance.pdf
Copied: fig5a_embedding_variance.png
Copied: fig5b_intrinsic_structure.pdf
Copied: fig5b_intrinsic_structure.png
Copied: figS3_distance_distortion.pdf
Copied: figS3_distance_distortion.png
Copied: fig_main_mi_control.pdf
Copied: fig_main_mi_control.png
All figures saved to: /content/drive/MyDrive/pe_experiment/figures_revision
